In [24]:
# On installe les librairies nécessaires pour :
# - charger un modèle de langage (transformers)
# - gérer les tokenizers (sentencepiece est requis par T5)
# - accélérer l'exécution si un GPU est disponible (accelerate)

!pip -q install transformers sentencepiece accelerate

In [25]:
# torch : backend de calcul (CPU / GPU)
# AutoTokenizer : transforme le texte en tokens compréhensibles par le modèle
# AutoModelForSeq2SeqLM : modèle "texte → texte" (question → réponse)

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [26]:
# Nom du modèle choisi
# FLAN-T5 est un modèle :
# - instruction-following
# - léger
# - bien adapté à l’expérimentation pédagogique

MODEL_NAME = "google/flan-t5-base"

In [27]:
# Chargement du tokenizer (texte → tokens)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Chargement du modèle de génération (tokens → texte)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

In [28]:
# On vérifie si un GPU est disponible (Colab parfois)
# Sinon, le modèle tournera sur CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# On déplace le modèle sur le device choisi
model = model.to(device)

print("Device:", device)
print("Model:", MODEL_NAME)

Device: cpu
Model: google/flan-t5-base


In [33]:
def ask(prompt, max_new_tokens=128, temperature=0.0):
    # Cette fonction sert à :
    # 1) transformer ton texte (prompt) en tokens
    # 2) envoyer ces tokens au modèle
    # 3) récupérer la réponse en texte

    # 1) Texte -> tenseurs (PyTorch) que le modèle comprend
    # return_tensors="pt" : format PyTorch
    # truncation=True : coupe le prompt si trop long
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    # 2) Mettre les inputs sur le même "device" que le modèle (CPU ou GPU)
    # (plus fiable que .to(device) direct sur inputs)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 3) temperature > 0 => le modèle peut générer des réponses plus variées
    # temperature = 0 => réponse "stable" (souvent la même)
    do_sample = temperature > 0

    # 4) Générer la réponse (tokens de sortie)
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
    )

    # 5) Tokens -> texte lisible
    answer = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    return answer

In [34]:
prompt = "Explain what a supply chain is in 3 bullet points."
print(ask(prompt))

Supply chain is a chain of goods and services that are sold to consumers.


In [36]:
prompt = """Explain what a supply chain is.
Answer in exactly 3 bullet points.
Each bullet must be 12 to 18 words.
Use this format:
- ...
- ...
- ..."""
print(ask(prompt))

Supply chain is a chain of goods and services.


In [37]:
prompt = """Explain what a supply chain is.
Answer in exactly 3 bullet points:
- ...
- ...
- ..."""
print(ask(prompt, temperature=0.7))

Supply chain is the process of providing products.


In [38]:
prompt = """Write EXACTLY this:
- AAA
- BBB
- CCC
Nothing else."""
print(ask(prompt, temperature=0.7))


BBB
